# 10 — Training GSL/SGL su Colab (anti-overfitting) — I/O su Drive + smoke test

Notebook **driver** per lanciare, sul ramo `master`, la pipeline con **SGL/GSL loss**
(`--gsl`) e le **leve anti-overfitting** (early stopping, media mobile sul val
fg-Dice, weight decay, LR encoder frenato, augmentation geometrica DTM-safe).

**Riusa quello che hai gia' su Drive** (niente ricalcoli inutili):
- `data/processed_dataset.zip` — il dataset preprocessato;
- `data/gsl_class_weights.json` — i pesi globali GSL (accetta anche il nome `gls_...`);
- `data/dtms_archive/{train,val}_dtms.tar` — le DTM firmate gia' pre-calcolate.

E **salva tutto su Drive** dopo il training (nessuna perdita): checkpoint
(`best.pth`/`last.pth`), `history.json`, metriche 3D (`test_3d_metrics_*.json`),
grafici delle curve e tabella CSV finiscono in `gsl_runs/<RUN_NAME>/`.

Prima del run vero: **tre smoke test** crescenti (ambiente+loss, dataset,
run-in-miniatura). **Ordine:** esegui dall'alto verso il basso. Serve GPU.

## 0. Verifica GPU

In [ ]:
# Smoke test 0: senza GPU il training e' improponibile. Fermati subito se manca.
!nvidia-smi -L
import subprocess
gpu = subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True).stdout.strip()
assert gpu and 'GPU 0' in gpu, 'Nessuna GPU: Runtime -> Cambia tipo di runtime -> GPU'
print('GPU OK ->', gpu.splitlines()[0])

## 1. Monta Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 1b. Percorsi su Drive (input e output)

`DRIVE_BASE` e' la cartella che contiene `data/` (input) e dove verranno scritti
gli output. **Adegua solo questa riga** al tuo Drive. Sotto `data/` il notebook si
aspetta `processed_dataset.zip`, `gsl_class_weights.json` e
`dtms_archive/{train,val}_dtms.tar`.

In [ ]:
import os

DRIVE_BASE = '/content/drive/MyDrive/BraTS_Project'   # <-- la cartella che contiene 'data/'
DRIVE_DATA = os.path.join(DRIVE_BASE, 'data')          # input: dataset + pesi GSL + dtms_archive
DRIVE_OUT  = os.path.join(DRIVE_BASE, 'gsl_runs')      # output: checkpoint, metriche, grafici, csv

assert os.path.isdir(DRIVE_DATA), f'Cartella dati non trovata: {DRIVE_DATA} (adegua DRIVE_BASE)'
print('Input  <-', DRIVE_DATA)
print('Output ->', DRIVE_OUT)
print('  contenuto di data/:', sorted(os.listdir(DRIVE_DATA)))

## 2. Clona il repo (ramo `master`) e installa le dipendenze

In [ ]:
REPO_URL = 'https://github.com/Drastid/BraTS-PEDs-brain-tumour-segmentation.git'
BRANCH   = 'master'   # ramo principale (adegua se il tuo default e' 'main')
REPO_DIR = '/content/BraTS-PEDs-brain-tumour-segmentation'

if not os.path.isdir(REPO_DIR):
    !git clone $REPO_URL $REPO_DIR
%cd $REPO_DIR
!git fetch --all -q && (git checkout $BRANCH -q && git pull -q origin $BRANCH || echo "  (uso il branch di default del repo)")
!git log --oneline -1
!pip install -q -r requirements.txt
print('Repo pronto in', REPO_DIR)

## 3. Smoke test A — ambiente + forward della GSL loss

In pochi secondi: import ok, config accetta le leve anti-overfitting, e la
**GSL loss** fa un forward valido su un mini-batch finto. Se qui si rompe qualcosa
(versioni/import) lo scopri prima di toccare dataset e GPU.

In [ ]:
import torch
from src.config import TrainConfig
from src.losses import DiceFocalGSLLoss, AlphaScheduler, compute_global_class_weights

cfg = TrainConfig(early_stopping=True, es_patience=5, es_smooth_window=3,
                  weight_decay=5e-4, encoder_lr_mult=0.05,
                  augment_prob=0.6, augment_strength=1.5)
print('config OK -> early_stopping', cfg.early_stopping,
      '| es_smooth_window', cfg.es_smooth_window, '| aug_strength', cfg.augment_strength)

B, C, H, W = 2, cfg.num_classes, 32, 32
logits  = torch.randn(B, C, H, W)
targets = torch.randint(0, C, (B, H, W))
dtm     = torch.randn(B, C, H, W)
wk      = compute_global_class_weights([1e6, 1e4, 5e3, 2e3])
crit    = DiceFocalGSLLoss(num_classes=C, gsl_class_weights=wk,
                           scheduler=AlphaScheduler('step', total_epochs=30, step_length=5))
crit.set_epoch(0)
total, region, gsl, alpha = crit(logits, targets, dtm)
assert torch.isfinite(total), 'GSL loss non finita'
print(f'GSL loss OK -> total={total.item():.4f} region={region.item():.4f} '
      f'gsl={gsl.item():.4f} alpha={float(alpha):.2f}')

## 4. Porta il dataset da Drive su disco locale (NVMe)

Copia `data/processed_dataset.zip` da Drive **una volta** e lo scompatta in
`/content/`. Salta se gia' presente.

In [ ]:
DATA_ROOT   = '/content/processed_dataset'
DATASET_ZIP = os.path.join(DRIVE_DATA, 'processed_dataset.zip')

if not os.path.isdir(DATA_ROOT):
    assert os.path.isfile(DATASET_ZIP), f'Zip non trovato: {DATASET_ZIP}'
    !cp "$DATASET_ZIP" /content/
    !unzip -q /content/processed_dataset.zip -d /content/
assert os.path.isdir(os.path.join(DATA_ROOT, 'train', 'images')), 'processed_dataset/train/images mancante'
print('Dataset pronto in', DATA_ROOT)

## 5. Smoke test B — struttura e shape del dataset

Le split ci sono e uno slice ha il formato atteso: immagine `[4, H, W]` float32
(4 modalita' MRI), maschera `[H, W]` con label in {0,1,2,3}.

In [ ]:
import glob
import numpy as np

for split in ['train', 'val', 'test']:
    d = os.path.join(DATA_ROOT, split, 'images')
    n = len(glob.glob(os.path.join(d, '*.npy'))) if os.path.isdir(d) else 0
    print(f'  {split:5s}: {n} slice')
    assert n > 0 or split == 'test', f'split {split} vuota'

img_f = sorted(glob.glob(os.path.join(DATA_ROOT, 'train', 'images', '*.npy')))[0]
msk_f = os.path.join(DATA_ROOT, 'train', 'masks', os.path.basename(img_f))
img, msk = np.load(img_f), np.load(msk_f)
print('  immagine:', img.shape, img.dtype, '| maschera:', msk.shape, msk.dtype, '| label:', np.unique(msk))
assert img.ndim == 3 and img.shape[0] == 4, 'attese 4 modalita [4,H,W]'
assert set(np.unique(msk)).issubset({0, 1, 2, 3}), 'label fuori da {0,1,2,3}'
print('Dataset OK')

## 6. Ripristina i dati GSL da Drive (pesi globali + DTM) — niente ricalcolo

Invece di ricalcolare pesi e DTM (lento), li **prendiamo da Drive**:
- copia `gsl_class_weights.json` (accetta anche il nome `gls_class_weights.json`) in `DATA_ROOT`;
- estrae `data/dtms_archive/{train,val}_dtms.tar` dentro `<split>/` (i tar contengono `dtms/*.npy`,
  quindi si ottiene `train/dtms/` e `val/dtms/`).

Le DTM di **test non servono** (la valutazione 3D non le usa). Se su Drive manca
qualcosa, la cella **ripiega** su `precompute_gsl_stats` per generare cio' che manca.

In [ ]:
import sys, shutil

WEIGHTS_JSON = os.path.join(DATA_ROOT, 'gsl_class_weights.json')
DTM_ARCHIVE  = os.path.join(DRIVE_DATA, 'dtms_archive')

def _have_gsl():
    return (os.path.isfile(WEIGHTS_JSON)
            and glob.glob(os.path.join(DATA_ROOT, 'train', 'dtms', '*.npy'))
            and glob.glob(os.path.join(DATA_ROOT, 'val', 'dtms', '*.npy')))

if _have_gsl():
    print('[skip] dati GSL gia presenti in', DATA_ROOT)
else:
    # 1) pesi globali da Drive (normalizza il nome al canonico gsl_class_weights.json)
    cand = [os.path.join(DRIVE_DATA, n) for n in ('gsl_class_weights.json', 'gls_class_weights.json')]
    srcw = next((p for p in cand if os.path.isfile(p)), None)
    if srcw:
        shutil.copy2(srcw, WEIGHTS_JSON)
        print('  pesi GSL da Drive:', os.path.basename(srcw), '->', WEIGHTS_JSON)

    # 2) DTM: estrai i tar da Drive dentro <split>/  (contengono dtms/*.npy)
    for split in ['train', 'val']:
        tar = os.path.join(DTM_ARCHIVE, f'{split}_dtms.tar')
        if os.path.isfile(tar):
            dest = os.path.join(DATA_ROOT, split)
            os.makedirs(dest, exist_ok=True)
            print(f'  estraggo {os.path.basename(tar)} -> {os.path.join(dest, "dtms")}')
            subprocess.run(['tar', '-xf', tar, '-C', dest], check=True)
        else:
            print(f'  [warn] archivio mancante su Drive: {tar}')

    # 3) fallback: se ancora incompleto, ricalcola in loco
    if not _have_gsl():
        print('  [fallback] dati GSL incompleti su Drive -> ricalcolo con precompute_gsl_stats')
        subprocess.run([sys.executable, '-m', 'scripts.precompute_gsl_stats',
                        '--data-root', DATA_ROOT, '--splits', 'train', 'val'],
                       cwd=REPO_DIR, check=True)

assert _have_gsl(), 'Dati GSL non disponibili (ne su Drive ne ricalcolati)'
import json as _json
print('GSL pronta -> pesi', _json.load(open(WEIGHTS_JSON)).get('weights'),
      '| DTM train:', len(glob.glob(os.path.join(DATA_ROOT, 'train', 'dtms', '*.npy'))),
      '| DTM val:',   len(glob.glob(os.path.join(DATA_ROOT, 'val', 'dtms', '*.npy'))))

## 7. Configurazione del run

`RUN_NAME` isola gli output su Drive (`gsl_runs/<RUN_NAME>/`) e i checkpoint locali:
non sovrascrivi il training precedente. `MODELS` = architetture (inizia da `unet`).
`ANTIOF` = leve anti-overfitting, modificabili. `--augment-intensity` non e' incluso:
corromperebbe la DTM, quindi con la GSL la pipeline lo ignora.

In [ ]:
CKPT_ROOT  = '/content/checkpoints'          # locale (veloce); il backup va su Drive
BACKUP_DIR = DRIVE_OUT                         # su Drive: checkpoint + metriche 3D
RUN_NAME   = 'gsl_antiof_unet'
OUT_DIR    = os.path.join(DRIVE_OUT, RUN_NAME) # su Drive: grafici + csv del run
MODELS     = ['unet']                          # es. ['unet', 'fpn', 'segformer']
APPLY_A100 = True                              # resnet50 / mit-b2 / crop224 / bf16

GSL_ALPHA_MIN = 0.2                            # floor di alpha in Phase 2

ANTIOF = [
    '--early-stopping', '--es-patience', '5', '--es-smooth-window', '3',
    '--weight-decay', '5e-4',
    '--encoder-lr-mult', '0.05',
    '--augment-prob', '0.6', '--augment-strength', '1.5',
]
os.makedirs(OUT_DIR, exist_ok=True)
print('Run:', RUN_NAME, '| modelli:', MODELS, '| alpha_min:', GSL_ALPHA_MIN)
print('Output del run su Drive ->', OUT_DIR)
print('Leve anti-overfitting:', ' '.join(ANTIOF))

## 8. Smoke test C — run end-to-end in miniatura (`--smoke-test`)

`run_pipeline.py --smoke-test` (encoder leggero, **1+1 epoche**) con GSL loss e
early stopping, isolato in `run-name=smoke_gsl`, `--no-eval` per restare veloce.
Verifica l'intera catena: DataLoader con DTM, GSL schedulata, early stopping,
salvataggio checkpoint. Se passa, il run vero non dovrebbe sorprenderti.

In [ ]:
import time

smoke_cmd = ['python', 'run_pipeline.py', '--smoke-test',
             '--data-root', DATA_ROOT, '--ckpt-root', CKPT_ROOT,
             '--run-name', 'smoke_gsl', '--models', 'unet',
             '--gsl', '--gsl-alpha-min', str(GSL_ALPHA_MIN),
             '--early-stopping', '--es-patience', '5', '--es-smooth-window', '3',
             '--no-eval']
print('  $ ' + ' '.join(smoke_cmd)); print('=' * 80)
t0 = time.time()
proc = subprocess.Popen(smoke_cmd, cwd=REPO_DIR, stdout=subprocess.PIPE,
                        stderr=subprocess.STDOUT, text=True)
for line in proc.stdout:
    print(line, end='')
proc.wait()
assert proc.returncode == 0, f'Smoke C fallito (codice {proc.returncode})'
best = os.path.join(CKPT_ROOT, 'smoke_gsl', 'unet', 'best.pth')
assert os.path.isfile(best), f'best.pth non salvato: {best}'
print(f'\nSmoke C OK in {(time.time()-t0)/60:.1f} min -> {best}')

## 9. Run vero — GSL loss + anti-overfitting + valutazione 3D (backup su Drive)

Training completo con `--apply-a100-config`, `--gsl`, le leve `ANTIOF`, poi
`--evaluate` (Dice/IoU/HD95 sul test). Con `--backup-dir` su Drive, checkpoint
(`best.pth`/`last.pth`/`history.json`) e metriche 3D vengono copiati su Drive a
fine training. Cambia `RUN_NAME` (cella 7) per un nuovo run senza sovrascrivere.

In [ ]:
cmd = ['python', 'run_pipeline.py']
if APPLY_A100:
    cmd.append('--apply-a100-config')
cmd += ['--data-root', DATA_ROOT, '--ckpt-root', CKPT_ROOT,
        '--run-name', RUN_NAME, '--models', *MODELS,
        '--gsl', '--gsl-alpha-min', str(GSL_ALPHA_MIN),
        *ANTIOF,
        '--evaluate', '--backup-dir', BACKUP_DIR]

print('  $ ' + ' '.join(cmd)); print('=' * 80)
t0 = time.time()
proc = subprocess.Popen(cmd, cwd=REPO_DIR, stdout=subprocess.PIPE,
                        stderr=subprocess.STDOUT, text=True)
for line in proc.stdout:
    print(line, end='')
proc.wait()
if proc.returncode != 0:
    print(f'\n[ERRORE] run uscito con codice {proc.returncode}')
else:
    print(f'\n[ok] run {RUN_NAME} completato in {(time.time()-t0)/60:.1f} min')
    print('  checkpoint+metriche su Drive ->', os.path.join(BACKUP_DIR, RUN_NAME))

## 10. Risultati: best val fg-Dice, gap train-val, metriche 3D

Per ogni modello: best val fg-Dice (+ epoca), gap train-val a quell'epoca, e
metriche 3D sul test (Dice/IoU foreground + HD95 ET). Legge la `history.json` del
run e il JSON di `evaluate_3d_test`.

In [ ]:
import pandas as pd
import json

EVAL_ROOT = os.path.join(REPO_DIR, 'evaluation_outputs', RUN_NAME)

def best_from_history(arch):
    hp = os.path.join(CKPT_ROOT, RUN_NAME, arch, 'history.json')
    if not os.path.isfile(hp):
        return None
    rows = json.load(open(hp))
    best = max(rows, key=lambda r: r.get('val_fg_dice', -1))
    tr, va = best.get('train_fg_dice'), best.get('val_fg_dice')
    return {'best_epoch': best.get('epoch'), 'best_val_fg': va,
            'train_fg_at_best': tr, 'gap_train_val': (tr - va) if (tr and va) else None,
            'n_epochs': len(rows)}

def test_3d(arch):
    p = os.path.join(EVAL_ROOT, f'test_3d_metrics_{arch}.json')
    if not os.path.isfile(p):
        return {}
    s = json.load(open(p)).get('summary', {})
    return {'test_fg_dice': s.get('mean_fg_dice'), 'test_fg_iou': s.get('mean_fg_iou'),
            'hd95_ET': s.get('ET', {}).get('hd95_mean')}

rows = []
for arch in MODELS:
    h = best_from_history(arch)
    if h is None:
        print(f'  [warn] nessuna history per {arch}'); continue
    rows.append({'model': arch, **h, **test_3d(arch)})

df = pd.DataFrame(rows)
for c in ['best_val_fg', 'train_fg_at_best', 'gap_train_val', 'test_fg_dice', 'test_fg_iou', 'hd95_ET']:
    if c in df: df[c] = df[c].astype(float).round(4)
df

## 11. Curve di training (train vs val fg-Dice) — salvate su Drive

Curve train/val fg-Dice per epoca: se la train sale mentre la val si appiattisce,
stai overfittando — ed e' li' che le leve intervengono. La linea verticale segna
il best checkpoint. Ogni grafico e' **salvato come PNG su Drive** in `OUT_DIR`.

In [ ]:
import matplotlib.pyplot as plt

for arch in MODELS:
    hp = os.path.join(CKPT_ROOT, RUN_NAME, arch, 'history.json')
    if not os.path.isfile(hp):
        continue
    rows = json.load(open(hp))
    ep  = [r['epoch'] for r in rows]
    trd = [r.get('train_fg_dice') for r in rows]
    vad = [r.get('val_fg_dice') for r in rows]
    best_ep = max(rows, key=lambda r: r.get('val_fg_dice', -1))['epoch']
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(ep, trd, '-o', ms=3, label='train fg-Dice')
    ax.plot(ep, vad, '-o', ms=3, label='val fg-Dice')
    ax.axvline(best_ep, color='grey', ls='--', alpha=0.6, label=f'best (ep {best_ep})')
    ax.set_xlabel('epoca'); ax.set_ylabel('fg-Dice')
    ax.set_title(f'{arch} - curve train/val ({RUN_NAME})')
    ax.legend(); ax.grid(alpha=0.3)
    plt.tight_layout()
    png = os.path.join(OUT_DIR, f'curves_{arch}.png')
    fig.savefig(png, dpi=140, bbox_inches='tight')   # <-- salva su Drive
    print('  grafico salvato ->', png)
    plt.show()

## 12. Salva su Drive tabella + history (nulla va perso)

Scrive la tabella dei risultati come CSV in `OUT_DIR` e copia anche le
`history.json` di ogni modello accanto ai grafici, cosi' in `gsl_runs/<RUN_NAME>/`
hai tutto: checkpoint, metriche 3D, grafici, history e CSV.

In [ ]:
from datetime import datetime
import shutil

os.makedirs(OUT_DIR, exist_ok=True)
stamp = datetime.now().strftime('%Y%m%d_%H%M%S')
csv_path = os.path.join(OUT_DIR, f'results_{stamp}.csv')
df.to_csv(csv_path, index=False)
print('CSV ->', csv_path)

for arch in MODELS:
    hp = os.path.join(CKPT_ROOT, RUN_NAME, arch, 'history.json')
    if os.path.isfile(hp):
        dst = os.path.join(OUT_DIR, f'history_{arch}.json')
        shutil.copy2(hp, dst)
        print('history ->', dst)

print('\nTutto salvato su Drive in', OUT_DIR)
print('  (checkpoint best/last + metriche 3D sono gia stati copiati dal --backup-dir)')